# 🎵 Train your own IPT model — Google Colab

Train an **Instrumental Playing Techniques** classifier in the browser, with no local install,
and get a `.ts` model for `ipt~` / `pipo.ipt` in Max.

**Before you start**
1. Set the runtime to GPU: `Runtime` ▸ `Change runtime type` ▸ **T4 GPU** (or any GPU).
2. Put your audio in a folder of your Google Drive, with one sub-folder per playing technique.
   The sub-folder names become the class names of your model.

```
my_data_folder/
├── ordinario/
│   ├── file1.wav
│   └── file2.wav
└── staccato/
    └── file1.wav
```

Then run the three cells below, from top to bottom. ▶️

## 1. Install

In [ ]:
#@title Get the code and install it (about 2 min)
import os

REPO_URL = "https://github.com/DYCI2/ipt_recognition.git"
BRANCH   = "dev-performance"  #@param {type:"string"}

if not os.path.isdir("/content/ipt_recognition"):
    !git clone -q --branch {BRANCH} {REPO_URL} /content/ipt_recognition
%cd /content/ipt_recognition

# Colab's default runtime ships Python 3.13, but this project pins torch==2.2.2,
# numba==0.60.0, etc., which only exist up to Python 3.12. We therefore create a separate
# Python 3.11 environment with uv (it downloads Python itself) and run all the scripts with
# it. Keeping torch 2.2.2 also guarantees the exported .ts loads in ipt~.
!pip install -q uv
!uv venv -q --allow-existing --python 3.11 /content/ipt_env
!uv pip install -q --python /content/ipt_env/bin/python -r requirements.txt

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "none, training will be slow (Runtime > Change runtime type > GPU)")

## 2. Settings

- `scratch`: train a new `base` model on your data only.
- `flute`: fine-tune the pretrained flute model on your classes.
- `guitar`: fine-tune the pretrained electric guitar model on your classes.

Fine-tuning keeps what the pretrained model learned about sound and learns your own class
names, so you can use any number of classes with any names. It usually needs less data than
training from scratch.

In [ ]:
#@title Settings
#@markdown Folder in your Google Drive that holds one sub-folder per class:
drive_data_path = "/content/drive/MyDrive/my_data_folder"  #@param {type:"string"}
#@markdown Name of this run (use a new one for every run):
run_name = "my_run"  #@param {type:"string"}
#@markdown How to start:
start_from = "scratch"  #@param ["scratch", "flute", "guitar"]
#@markdown Maximum number of training loops (it stops earlier when it stops improving):
epochs = 100  #@param {type:"integer"}

#@markdown ---
#@markdown Only used with `scratch` (fine-tuning always uses the pretrained model's values):
sr = 44100  #@param {type:"integer"}
#@markdown Classification window, in `ms` or `samps`. Longer suits slow techniques (e.g. legato).
segment_length = "14700 samps"  #@param {type:"string"}

#@markdown ---
#@markdown Folder in your Google Drive where the trained model is saved:
drive_output_dir = "/content/drive/MyDrive/ipt_runs"  #@param {type:"string"}

## 3. Train

In [ ]:
#@title Prepare the data, train, and save the model to Google Drive
import glob, os, shutil, torch
from google.colab import drive

if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
%cd /content/ipt_recognition

# Audio settings of the pretrained models; the new data is prepared with the same ones
PRETRAINED = {
    'flute':  dict(pretrained='pretrained/flute_pretrained.pth', model='base',
                   sr=44100, seglen='14700 samps', n_mels=384, hop_length=128, f_min=120),
    'guitar': dict(pretrained='pretrained/eguitar_help.ts', model='eguitar',
                   sr=8000, seglen='7168 samps', n_mels=128, hop_length=512, f_min=150),
}
if start_from == 'scratch':
    # base expects a 384 x 115 log-mel image (384 mels, 14700 samples at hop 128):
    # the hop is scaled with the window so that any window length gives the same image
    value, unit = segment_length.split()
    assert unit in ('ms', 'samps'), "segment_length must be like '500 ms' or '14700 samps'"
    n_samples = int(value) if unit == 'samps' else int(int(value) / 1000 * sr)
    assert n_samples >= 114, 'segment_length is too short'
    cfg = dict(pretrained=None, model='base', sr=sr, seglen=segment_length, n_mels=384,
               hop_length=n_samples // 114, f_min=20)
    if cfg['hop_length'] > 512:
        print('Warning: a window this long makes the hop larger than the smallest FFT (512), '
              'part of the audio is skipped by that analysis. Use a shorter window or a lower sampling rate.')
else:
    cfg = PRETRAINED[start_from]
pretrained, model, sr_run, seglen = cfg['pretrained'], cfg['model'], cfg['sr'], cfg['seglen']
n_mels, hop_length, f_min = cfg['n_mels'], cfg['hop_length'], cfg['f_min']

# Copy the audio folder from Drive into data/raw/
assert os.path.isdir(drive_data_path), f'Not found in your Drive: {drive_data_path}'
assert not os.path.exists(os.path.join('data', 'preprocessed', run_name)), \
    f'run_name {run_name!r} was already used in this session, choose a new one'
data_folder = os.path.basename(os.path.normpath(drive_data_path))
data_path = os.path.join('data', 'raw', data_folder)
if os.path.isdir(data_path):
    shutil.rmtree(data_path)
shutil.copytree(drive_data_path, data_path)

audio_ext = ('.wav', '.mp3', '.flac', '.aiff', '.aif')
classes = {}
for root, _, files in os.walk(data_path):
    n = sum(f.lower().endswith(audio_ext) for f in files)
    if n and root != data_path:
        classes[os.path.basename(root)] = classes.get(os.path.basename(root), 0) + n
assert len(classes) >= 2, 'Put at least two class sub-folders with audio files in your data folder'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Start from: {start_from}  (architecture {model}), device {device}')
print(f'Audio: {sr_run} Hz, window {seglen}, {n_mels} mels, hop {hop_length}')
print(f'{len(classes)} classes:')
for name, n in sorted(classes.items()):
    print(f'  {name}: {n} file(s)')

# 20 % of the files of each class are kept aside to check the model during training
PY = 'MPLBACKEND=Agg /content/ipt_env/bin/python'
!{PY} preprocess.py --name {run_name} --train_dir {data_folder} --sampling_rate {sr_run} -seglen "{seglen}"
if start_from == 'scratch':
    !{PY} train.py --name {run_name} --model {model} --sampling_rate {sr_run} -seglen "{seglen}" --n_mels {n_mels} --hop_length {hop_length} --f_min {f_min} --device {device} --epochs {epochs} --early_stopping 10
else:
    !{PY} finetune.py --name {run_name} --pretrained {pretrained} --model {model} --sampling_rate {sr_run} -seglen "{seglen}" --n_mels {n_mels} --hop_length {hop_length} --f_min {f_min} --device {device} --epochs {epochs} --early_stopping 10

# Save the run to Google Drive
runs = sorted(glob.glob(f'runs/{run_name}_*'), key=os.path.getmtime)
assert runs and glob.glob(os.path.join(runs[-1], '*.ts')), 'No model found. Did the training finish? Read the log above.'
os.makedirs(drive_output_dir, exist_ok=True)
out = os.path.join(drive_output_dir, os.path.basename(runs[-1]))
if os.path.isdir(out):
    shutil.rmtree(out)
shutil.copytree(runs[-1], out)
ts = glob.glob(os.path.join(out, '*.ts'))[0]
m = torch.jit.load(ts)
print('\nModel saved to your Drive:', ts)
print('Classes:', m.get_classnames())
print(f'Sampling rate: {m.get_sr()} Hz, window: {m.get_seglen()} samples')
print('Load the .ts file in Max with ipt~ or pipo.ipt.')